In [26]:
with open("data/input.txt", "r", encoding = "utf-8") as f:
    text = f.read()


In [27]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [28]:
chars = sorted(list(set(text)))
print(chars)
vocab_size = len(chars)
print(vocab_size)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
65


In [29]:
stoi = {char : i for i,char in enumerate(chars)}
itos = {i : char for i,char in enumerate(chars)}
encoder = lambda s: [stoi[c] for c in s]
decoder = lambda x: ''.join([itos[i] for i in x])
print(encoder("hello"))
print(decoder([46, 43, 50, 50, 53]))

[46, 43, 50, 50, 53]
hello


In [30]:
import torch
data = torch.tensor(encoder(text), dtype=torch.long)
print(data.shape)
print(data[:1000])

torch.Size([1115394])
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
        47, 59, 57

In [31]:
split = int(len(data)*0.9)
train = data[:split]
test = data[split:]


In [32]:
# transformers are trained on randomly chose chunks/blocks of training set with some max length(context length)
block_size = 8
print(train[:block_size + 1])

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])


In [33]:
# we want the model to look at a block and say when given 18, predict 47; when given 18 + 47, predict 56; when given 18 + 47 + 56
# predict 57, and so on. It looks like such
x = train[:block_size]
y = train[1:block_size + 1]
for t in range(block_size):
    context_length = x[:t+1]
    target = y[t]
    print(f"when input is {context_length} the target is {target}")


when input is tensor([18]) the target is 47
when input is tensor([18, 47]) the target is 56
when input is tensor([18, 47, 56]) the target is 57
when input is tensor([18, 47, 56, 57]) the target is 58
when input is tensor([18, 47, 56, 57, 58]) the target is 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target is 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [34]:
torch.manual_seed(1337) 
batch_size = 4 #independent sequences to process in parrallel
def get_batch(split):
    data = train if split == 'train' else test
    ix = torch.randint(len(data) - block_size, (batch_size,)) # generating batch_size numbers between 0 and len(data) - block_size
    x = torch.stack([data[i:i+block_size] for i in ix]) # get the context length starting at i in ix for all numbers in ix. stack them in to a tensor
    # tensor is now a batch_size x block_size matrix(each row is a block)
    y = torch.stack([data[i+1:i+1+block_size] for i in ix]) # same thing with an offset of 1 
    return x, y
x,y = get_batch('train')
print('inputs:')
print(x.shape)
print(x)
print('targets')
print(y.shape)
print(y)

for b in range(batch_size):
    for t in range(block_size):
        context = x[b, :t+1]
        target = y[b, t]
        print(f"when input is {context.tolist()}, target is {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
when input is [24], target is 43
when input is [24, 43], target is 58
when input is [24, 43, 58], target is 5
when input is [24, 43, 58, 5], target is 57
when input is [24, 43, 58, 5, 57], target is 1
when input is [24, 43, 58, 5, 57, 1], target is 46
when input is [24, 43, 58, 5, 57, 1, 46], target is 43
when input is [24, 43, 58, 5, 57, 1, 46, 43], target is 39
when input is [44], target is 53
when input is [44, 53], target is 56
when input is [44, 53, 56], target is 1
when input is [44, 53, 56, 1], target is 58
when input is [44, 53, 56, 1, 58], target is 46
when input is [44, 53, 56, 1, 58, 46], t

In [35]:
import torch.nn as nn
from torch.nn import functional as F
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off logits for the next token in from a look up table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
        
    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) #batch x time x channel tensor(4 x 8 x 65) = (batch x blocks x vocab_size)
        if targets is None:
            loss = None
        else: 
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_token): #idx = (Batch, Time) array of indices in the current context
        for _ in range(max_new_token):
            logits, loss = self(idx) #predictions
            logits = logits[:, -1, :] #(Batch, Channel)
            probs = F.softmax(logits, dim = 1)
            idx_next = torch.multinomial(probs, num_samples=1) #sample from distribution, (B, 1)
            idx = torch.cat((idx, idx_next), dim=1) # append sample to sequence, (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size) 
out, loss = m(x, y)
print(out.shape)
print(loss)

print(decoder(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_token=100)[0].tolist()))

torch.Size([32, 65])
tensor(5.0364, grad_fn=<NllLossBackward0>)

lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheH


In [36]:
optimizer = torch.optim.AdamW(m.parameters(), lr = 1e-3)
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())
print(decoder(m.generate(torch.zeros((1,1), dtype=torch.long), max_new_token=700)[0].tolist()))

2.362441062927246

M:
IUSh t,
F th he d ke alved.
Thupld, cipbll t
I: ir w, l me sie hend lor ito'l an e

I:
Gochosen ea ar btamandd halind
Aust, plt t wadyotl
I bel qunganonoth he m he de avellis k'l, tond soran:

WI he toust are bot g e n t s d je hid t his IAces I my ig t
Ril'swoll e pupat inouleacends-athiqu heamer te
Wht s

MI wect!-lltherotheve t fe;
WAnd py;

PO t s ld tathat, ir V
IO thesecin teot tit ado ilorer.
Ply, d'stacoes, ld omat mealellly yererer EMEvesas ie IZEd pave mautoofareanerllleyomerer but?
The t,
Ith'dwitile w? beren to'd ff a atrts brey s

ESesenther:
Ithon f at par,
NTmamy an flictong m




Mameld h che IN: an y is aslo'daDut, t thethiceve fur t anowik
Wirghe f bot-d at'prd
Anoper so


In [37]:
torch.manual_seed(1337)
B,T,C = 4, 8, 2
x = torch.rand(B, T, C)
x.shape
#we want to predict the char in the i-1th position based on a feature vector that contains the average of the previous i entries

torch.Size([4, 8, 2])

In [38]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1]
        xbow[b,t] = torch.mean(xprev, 0)
        
#making more efficient

In [39]:
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True) # each row adds up + 1, each entry in row i = 1/i
xbow2 = wei @ x #(B, T, T) @ (B, T, C) = (B,T,C) -> wei is T,T but pytorch broadcasts the B by working right to left from x
#xbow = xbow2...same idea optimized via matmul

In [40]:
#same process but using softmax
lt = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(lt == 0, float('-inf')) #where the entry is 0, replace with -inf
wei = F.softmax(wei, dim= -1) #apply softmax to each entry
xbow3 = wei @ x
print(xbow2)
print(xbow3)
#intro to soft attention...allows each token to interact with each other...aggregate via matmul

tensor([[[0.0783, 0.4956],
         [0.3507, 0.4590],
         [0.3006, 0.3156],
         [0.3717, 0.4108],
         [0.3326, 0.3806],
         [0.3953, 0.4140],
         [0.3470, 0.4644],
         [0.4134, 0.4368]],

        [[0.6005, 0.7079],
         [0.5554, 0.5572],
         [0.6657, 0.4908],
         [0.7234, 0.6090],
         [0.5817, 0.6344],
         [0.6161, 0.6865],
         [0.5946, 0.7081],
         [0.5362, 0.6462]],

        [[0.2944, 0.3677],
         [0.3887, 0.5215],
         [0.4333, 0.5322],
         [0.4675, 0.4611],
         [0.4948, 0.5105],
         [0.4255, 0.5525],
         [0.4748, 0.4842],
         [0.4875, 0.5094]],

        [[0.9100, 0.7684],
         [0.8118, 0.4135],
         [0.7959, 0.5978],
         [0.8454, 0.6482],
         [0.6993, 0.6530],
         [0.5973, 0.6449],
         [0.5726, 0.5670],
         [0.5115, 0.5632]]])
tensor([[[0.0783, 0.4956],
         [0.3507, 0.4590],
         [0.3006, 0.3156],
         [0.3717, 0.4108],
         [0.3326, 0.

In [ ]:
torch.manual_seed(1337)
B,T,C = 4, 8, 32
x = torch.randn(B,T,C)
head_size = 16
key = nn.Linear(C, head_size, bias= False)
query = nn.Linear(C,head_size, bias = False)
k = key(x) # 4 x 8 x 16
q = query(x)
value = nn.Linear(C, head_size, bias=False)
wei = q @ k.transpose(-2,-1) # matmul ignores all dims but last 2 (B,T, 16) @(B, 16, T) = (B, T, T)

tril = torch.tril(torch.ones(T,T))
# print(wei)
# print(tril)
wei = wei.masked_fill(tril == 0, float('-inf'))
#print(wei)
wei = F.softmax(wei, dim = -1)
v = value(x)
out = wei @ v
#need to scale by 1/sqrt(head_size) before applying softmax
out.shape

torch.Size([4, 8, 16])

In [50]:
wei

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
         [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
         [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
         [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1687, 0.8313, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2477, 0.0514, 0.7008, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4410, 0.0957, 0.3747, 0.0887, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0069, 0.0456, 0.0300, 0.7748, 0.1427, 0.0000, 0.0000, 0.0000],
         [0.0660, 0.089